# Day 03: Transformer Blocks & Attention Deep Dive
> *Inference Engineering* — Chapter 2.2.2–2.2.3 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime
**Prerequisite:** Day 01 (tokenization, KV cache basics)

**Topics:** Embedding layer, transformer block structure, feed-forward networks, multi-head attention, self-attention vs cross-attention, causal masking

## Overview

Day 01 covered the inference loop: tokenize, prefill, decode, sample. But what
happens *inside* the model during a forward pass? This notebook opens up that black box.

An LLM is a pipeline of three stages:

1. **Embedding layer** — converts token IDs into vectors
2. **Transformer blocks** (repeated N times) — each block refines the vectors using attention + feed-forward
3. **Output layer (LMHead)** — converts vectors back into logits over the vocabulary

Think of it like a multi-stage data pipeline: raw input goes in, gets transformed
by a series of identical processing stages, and a final stage produces the output.
The number of stages (layers) is one of the key architecture decisions — GPT-2 has 12,
Llama-3-70B has 80.

This notebook builds each stage from scratch in numpy/PyTorch so you can see
the data flow and the shapes at every step.

In [ ]:
# Setup
!pip install -q numpy torch matplotlib

import numpy as np
import matplotlib.pyplot as plt
import math

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | device: {DEVICE}")
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available -- numpy-only mode")

plt.rcParams.update({
    "figure.facecolor": "#0d1117", "axes.facecolor": "#161b22",
    "axes.edgecolor": "#30363d",   "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",  "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",      "grid.color": "#21262d",
})
%matplotlib inline

## Part 1: The Embedding Layer

Tokens are integers. Neural networks work with vectors of floating-point numbers.
The **embedding layer** is the lookup table that bridges these two worlds.

Each token ID maps to a vector of size `d_model` (the model's hidden dimension).
This vector is called an **embedding** — it's the token's representation in the model's
internal coordinate system.

Think of it as a key-value store: `token_id -> float_vector[d_model]`.
The values are learned during training.

In [ ]:
# Build an embedding layer from scratch

vocab_size = 50257   # GPT-2's vocabulary
d_model    = 768     # GPT-2's hidden dimension

# The embedding table: one row per token, each row is a d_model-dimensional vector
# In a real model, these values are learned during training
np.random.seed(42)
embedding_table = np.random.randn(vocab_size, d_model) * 0.02  # small init values

print(f"Embedding table shape: {embedding_table.shape}")
print(f"  = [{vocab_size} tokens] x [{d_model} dimensions per token]")
print(f"  = {embedding_table.nbytes / 1e6:.1f} MB of parameters\n")

# Look up embeddings for a short sequence
token_ids = [464, 2068, 7586, 21831]  # "The quick brown fox" in GPT-2
tokens    = ["The", "quick", "brown", "fox"]

embeddings = embedding_table[token_ids]  # just an array index -- no math
print(f"Input token IDs: {token_ids}")
print(f"Output shape:    {embeddings.shape}  = [seq_len=4, d_model={d_model}]")
print(f"\nEach token is now a vector of {d_model} numbers:")
for tok, emb in zip(tokens, embeddings):
    print(f"  '{tok:6s}' -> [{emb[0]:+.4f}, {emb[1]:+.4f}, {emb[2]:+.4f}, ... {emb[-1]:+.4f}]")

print(f"\nThese vectors flow into the first transformer block.")

## Part 2: Inside a Transformer Block

Each transformer block has two main sub-layers, run in sequence:

```
input
  │
  ├──> [Layer Norm] ─> [Multi-Head Attention] ─> + (residual)
  │                                              │
  ├──> [Layer Norm] ─> [Feed-Forward Network]  ─> + (residual)
  │                                              │
  output
```

The **residual connections** (the `+`) are like skip connections in a network — the input
is added back to the output. This lets the model learn "what to add" rather than
"what the full output should be", which makes training much easier on deep networks.

*Book Figure 2.7 shows the original transformer block diagram from "Attention Is All You Need" (Vaswani et al., 2017).*

### Where the parameters live

The book notes that the **feed-forward network** (FFN) holds the majority of the model's
trainable weights, while **attention** is the second-largest. Normalization and activation
functions are negligible in size.

Let's build each piece and see the shapes.

In [ ]:
# Build a transformer block from scratch using PyTorch
# We'll trace the data shapes through each sub-layer

d_model   = 768    # hidden dimension (GPT-2)
n_heads   = 12     # number of attention heads
d_ff      = 3072   # feed-forward inner dimension (typically 4x d_model)
seq_len   = 4      # our short sequence
head_dim  = d_model // n_heads  # 64 per head

print("=== Transformer Block Parameters (GPT-2 scale) ===")
print(f"d_model={d_model}, n_heads={n_heads}, head_dim={head_dim}, d_ff={d_ff}\n")

# Simulate input: [batch=1, seq_len=4, d_model=768]
torch.manual_seed(42)
x = torch.randn(1, seq_len, d_model)
print(f"Input shape: {list(x.shape)}  = [batch, seq_len, d_model]")

# --- Sub-layer 1: Layer Norm + Multi-Head Attention + Residual ---
norm1 = nn.LayerNorm(d_model)
# Attention projections: Q, K, V each project d_model -> d_model
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model, bias=False)  # output projection

normed = norm1(x)
Q = W_q(normed)  # [1, 4, 768]
K = W_k(normed)
V = W_v(normed)

print(f"\n--- Sub-layer 1: Attention ---")
print(f"After LayerNorm:  {list(normed.shape)}")
print(f"Q, K, V each:     {list(Q.shape)}")

# Reshape for multi-head: [batch, seq_len, n_heads, head_dim] -> [batch, n_heads, seq_len, head_dim]
Q_heads = Q.view(1, seq_len, n_heads, head_dim).transpose(1, 2)
K_heads = K.view(1, seq_len, n_heads, head_dim).transpose(1, 2)
V_heads = V.view(1, seq_len, n_heads, head_dim).transpose(1, 2)
print(f"After head split: {list(Q_heads.shape)}  = [batch, n_heads, seq_len, head_dim]")

# Attention scores + causal mask
scores = (Q_heads @ K_heads.transpose(-2, -1)) / math.sqrt(head_dim)
mask = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)
scores = scores + mask
weights = torch.softmax(scores, dim=-1)
attn_out = weights @ V_heads  # [1, 12, 4, 64]

# Concatenate heads back and project
attn_out = attn_out.transpose(1, 2).contiguous().view(1, seq_len, d_model)
attn_out = W_o(attn_out)
x_after_attn = x + attn_out  # residual connection
print(f"Attention output:  {list(attn_out.shape)}")
print(f"After residual:    {list(x_after_attn.shape)}")

# --- Sub-layer 2: Layer Norm + Feed-Forward Network + Residual ---
norm2 = nn.LayerNorm(d_model)
ff_up   = nn.Linear(d_model, d_ff, bias=False)    # 768 -> 3072 (expand)
ff_down = nn.Linear(d_ff, d_model, bias=False)     # 3072 -> 768 (project back)

normed2 = norm2(x_after_attn)
ff_out  = ff_down(torch.relu(ff_up(normed2)))      # up-project, ReLU, down-project
x_after_ff = x_after_attn + ff_out                 # residual

print(f"\n--- Sub-layer 2: Feed-Forward ---")
print(f"FFN inner shape:   [1, {seq_len}, {d_ff}]  (4x expansion)")
print(f"FFN output:        {list(ff_out.shape)}")
print(f"Block output:      {list(x_after_ff.shape)}")

# Parameter count comparison
attn_params = sum(p.numel() for p in [W_q, W_k, W_v, W_o] for p in p.parameters())
ff_params   = sum(p.numel() for p in [ff_up, ff_down] for p in p.parameters())
print(f"\n--- Parameter distribution per block ---")
print(f"Attention (Q+K+V+O):  {attn_params:>12,} params  ({attn_params/(attn_params+ff_params)*100:.0f}%)")
print(f"Feed-forward (up+dn): {ff_params:>12,} params  ({ff_params/(attn_params+ff_params)*100:.0f}%)")
print(f"\nThe FFN holds ~2/3 of the weights. Attention is the complex operation, but FFN is the big one.")

## Part 3: Multi-Head Attention

Day 01 showed attention with a single query-key-value set. Real models run
**multiple attention heads in parallel** — each head gets its own Q, K, V projections
and can specialize in different kinds of relationships.

Think of it like having multiple monitoring dashboards, each watching a different aspect
of the same system. One head might track subject-verb agreement, another might track
pronoun references, another might track position.

Mechanically:
- The d_model dimension is **split** across heads: `head_dim = d_model / n_heads`
- Each head runs attention independently on its own slice
- Results are **concatenated** back to d_model and projected through W_o

The next cell visualizes what different heads attend to on the same input.

In [ ]:
# Visualize attention patterns across heads
# Each head learns different relationships between tokens

tokens_display = ["The", "server", "crashed", "due", "to", "memory"]
n_tokens = len(tokens_display)
n_show_heads = 4  # show 4 of 12 heads

# Simulate: create different attention patterns per head
np.random.seed(99)
fig, axes = plt.subplots(1, n_show_heads, figsize=(16, 4))

head_labels = [
    "Head 1: recent tokens",
    "Head 2: first token",
    "Head 3: spread across all",
    "Head 4: adjacent token",
]

for h, (ax, label) in enumerate(zip(axes, head_labels)):
    # Generate plausible attention patterns
    if h == 0:  # recency bias
        w = np.zeros((n_tokens, n_tokens))
        for i in range(n_tokens):
            for j in range(i+1):
                w[i, j] = np.exp(-0.8 * (i - j))
    elif h == 1:  # attend to first token (like a [CLS] token)
        w = np.zeros((n_tokens, n_tokens))
        for i in range(n_tokens):
            w[i, 0] = 3.0
            for j in range(1, i+1):
                w[i, j] = 0.2
    elif h == 2:  # spread evenly
        w = np.ones((n_tokens, n_tokens)) * 0.5
        w += np.random.randn(n_tokens, n_tokens) * 0.3
    else:  # attend to previous token
        w = np.zeros((n_tokens, n_tokens))
        for i in range(n_tokens):
            w[i, max(0, i-1)] = 3.0
            for j in range(i+1):
                w[i, j] += 0.1

    # Apply causal mask and normalize
    causal = np.triu(np.full((n_tokens, n_tokens), -1e9), k=1)
    w = w + causal
    exp_w = np.exp(w - w.max(axis=-1, keepdims=True))
    w_norm = exp_w / exp_w.sum(axis=-1, keepdims=True)

    im = ax.imshow(w_norm, cmap='viridis', vmin=0, vmax=1)
    ax.set_xticks(range(n_tokens))
    ax.set_yticks(range(n_tokens))
    ax.set_xticklabels(tokens_display, fontsize=8, rotation=45, ha='right')
    ax.set_yticklabels(tokens_display, fontsize=8)
    ax.set_title(label, color='#e6edf3', fontsize=9)
    if h == 0:
        ax.set_ylabel('Query (current token)', color='#e6edf3')
    ax.set_xlabel('Key (attends to)', color='#e6edf3')

plt.suptitle('Different attention heads learn different patterns\n(same input, same block, different heads)',
             color='#e6edf3', fontsize=11)
plt.tight_layout()
plt.show()
print("Each head runs attention on a (seq_len x head_dim) slice independently.")
print(f"GPT-2: {n_heads} heads x {head_dim} dims = {d_model} total dimensions.")
print(f"Llama-3-70B: 64 query heads, but only 8 KV heads (GQA -- saves KV cache memory).")

## Part 4: Self-Attention vs Cross-Attention

The book (Section 2.2.3) describes two types of attention:

- **Self-attention:** Q, K, and V all come from the same sequence. This is what LLMs use.
  Each token attends to other tokens in its own sequence.

- **Cross-attention:** Q comes from one sequence, K and V come from another. Used in
  encoder-decoder models (e.g., Whisper for audio transcription) and image generation
  (the image attends to the text prompt).

For LLM inference, it's always **causal self-attention**: a token can only attend
to itself and tokens *before* it (not future tokens). This is enforced by the
causal mask we've been using.

In [ ]:
# Visualize causal mask: what each token position can see

seq_len = 8
causal_mask = np.tril(np.ones((seq_len, seq_len)))  # lower triangle = 1 (can see)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Self-attention (causal)
ax1.imshow(causal_mask, cmap='Greens', vmin=0, vmax=1)
ax1.set_title('Causal Self-Attention (LLMs)\nGreen = can attend, dark = masked', color='#e6edf3')
ax1.set_xlabel('Key position', color='#e6edf3')
ax1.set_ylabel('Query position', color='#e6edf3')
for i in range(seq_len):
    for j in range(seq_len):
        color = 'white' if causal_mask[i,j] == 0 else 'black'
        ax1.text(j, i, int(causal_mask[i,j]), ha='center', va='center', fontsize=9, color=color)

# Cross-attention (full visibility)
cross_mask = np.ones((6, seq_len))  # decoder queries can attend to all encoder keys
ax2.imshow(cross_mask, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax2.set_title('Cross-Attention (encoder-decoder)\nDecoder queries attend to ALL encoder keys', color='#e6edf3')
ax2.set_xlabel('Encoder key position', color='#e6edf3')
ax2.set_ylabel('Decoder query position', color='#e6edf3')

plt.tight_layout()
plt.show()
print("LLMs use causal self-attention: token i can only see tokens 0..i")
print("This is what makes autoregressive generation work -- no peeking at future tokens.")
print("\nCross-attention is used in Whisper, Stable Diffusion, and other encoder-decoder models.")

## Part 5: The Output Layer (LMHead)

After passing through all N transformer blocks, the final hidden state is converted
into logits over the vocabulary. This is the **output layer** (also called LMHead).

It's a single linear layer: `hidden_state [d_model] -> logits [vocab_size]`

Let's trace the full pipeline end-to-end with GPT-2.

In [ ]:
# Trace data flow through a real GPT-2 model
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2", output_hidden_states=True)
model.eval()

prompt = "The server crashed due to"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.hidden_states  # tuple of [batch, seq_len, d_model] per layer

print(f"Prompt: '{prompt}'")
print(f"Token count: {inputs['input_ids'].shape[1]}")
print(f"\n=== Data flow through the model ===")
print(f"Embedding output:    {list(hidden_states[0].shape)}  (layer 0 = after embedding)")
for i in [1, 6, 12]:
    if i < len(hidden_states):
        print(f"After block {i:2d}:       {list(hidden_states[i].shape)}")
print(f"Final hidden state:  {list(hidden_states[-1].shape)}  (layer {len(hidden_states)-1})")
print(f"Logits (LMHead out): {list(outputs.logits.shape)}  = [batch, seq_len, vocab_size={model.config.vocab_size}]")

# What the model predicts next
next_logits = outputs.logits[0, -1, :]
top5 = torch.topk(next_logits, 5)
print(f"\nTop-5 next token predictions:")
for score, tid in zip(top5.values, top5.indices):
    print(f"  {repr(tokenizer.decode(tid)):15s}  logit={score:.2f}")

In [ ]:
# Visualize how the hidden state changes as it passes through layers
# Use cosine similarity: how much does each layer change the representation?

n_layers = len(hidden_states) - 1  # exclude embedding
similarities = []
for i in range(1, len(hidden_states)):
    prev = hidden_states[i-1][0, -1, :]  # last token's hidden state
    curr = hidden_states[i][0, -1, :]
    cos_sim = torch.nn.functional.cosine_similarity(prev.unsqueeze(0), curr.unsqueeze(0)).item()
    similarities.append(cos_sim)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, n_layers + 1), similarities, 'o-', color='#1f6feb', lw=2, markersize=6)
ax.set_xlabel('Layer', color='#e6edf3')
ax.set_ylabel('Cosine similarity with previous layer', color='#e6edf3')
ax.set_title('How much each transformer block changes the representation\n(higher = less change)',
             color='#e6edf3')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.axhline(1.0, color='#3fb950', ls='--', alpha=0.5, label='No change (identity)')
ax.legend(framealpha=0.2)
plt.tight_layout()
plt.show()
print("Early layers make bigger changes (learning basic features).")
print("Later layers make smaller adjustments (refining the prediction).")
print("This is why some inference optimizations skip or merge later layers.")

## Part 6: Architecture from config.json

Every model on HuggingFace includes a `config.json` that tells you everything
about its architecture. The book (Section 2.2.1) explains how to parse these.

For example, an architecture name like `Qwen3MoeForCausalLM` tells you:
- **Qwen3** — model family and version
- **Moe** — Mixture of Experts (covered in Day 03)
- **ForCausalLM** — causal language model (autoregressive, left-to-right)

In [ ]:
# Read the architecture from a real model config
config = model.config

print("=== GPT-2 Architecture (from config.json) ===")
print(f"Architecture:    {config.architectures}")
print(f"Vocab size:      {config.vocab_size:,}")
print(f"Context window:  {config.n_positions:,} tokens")
print(f"Hidden dim:      {config.n_embd} (d_model)")
print(f"Layers:          {config.n_layer}")
print(f"Attention heads: {config.n_head}")
print(f"Head dimension:  {config.n_embd // config.n_head}")
print(f"FFN inner dim:   {4 * config.n_embd} (4x expansion)")

# Compare with bigger models (public configs)
models = [
    ("GPT-2",         config.n_layer, config.n_head, config.n_embd, config.vocab_size),
    ("Llama-3.1-8B",  32, 32, 4096, 128256),
    ("Llama-3.1-70B", 80, 64, 8192, 128256),
    ("Qwen2.5-72B",   80, 64, 8192, 152064),
]

print(f"\n{'Model':<18} {'Layers':>8} {'Heads':>8} {'d_model':>8} {'Vocab':>10} {'~Params':>10}")
print("-" * 65)
for name, layers, heads, d, vocab in models:
    # Rough param estimate: embedding + N*(attention + FFN) + output
    emb = vocab * d
    attn_per_layer = 4 * d * d  # Q, K, V, O
    ff_per_layer = 2 * d * (4 * d)  # up + down
    total = emb + layers * (attn_per_layer + ff_per_layer) + emb
    print(f"{name:<18} {layers:>8} {heads:>8} {d:>8} {vocab:>10,} {total/1e9:>9.1f}B")

print(f"\nNote: actual param counts differ slightly due to biases, norms, and GQA.")
print(f"The key takeaway: more layers = deeper model = more compute per forward pass.")

## Experiments: Try These

1. **Layer-by-layer probing:** For each hidden layer in GPT-2, extract the last token's
   hidden state, project it through the LMHead, and check the top-5 predictions.
   At which layer does the model first predict the correct next token?

2. **Head pruning:** In the multi-head attention visualization, some heads have very
   uniform distributions (not very useful). Count how many heads have entropy > 90%
   of maximum. These are candidates for pruning.

3. **FFN size sweep:** The FFN expansion ratio is typically 4x. Calculate the parameter
   count for a 32-layer model with 2x, 4x, and 8x expansion. Plot the tradeoff
   between FFN size and total model parameters.

## Key Takeaways

- An LLM has three stages: **embedding** (token IDs → vectors), **N transformer blocks** (attention + FFN), and an **output layer** (vectors → logits).
- Each transformer block runs **multi-head attention** then a **feed-forward network**, each with a residual connection.
- The **FFN holds ~2/3 of the parameters** per block. Attention is the more complex operation but uses fewer weights.
- **Multi-head attention** splits the model dimension across heads that run in parallel. Each head can learn different token relationships.
- LLMs use **causal self-attention** — each token can only attend to previous tokens. The causal mask enforces this.
- Architecture details come from `config.json`. The key numbers: layers, heads, d_model, vocab_size.
- Next: Day 03 — Mixture of Experts (MoE) Routing.

## References

- *Inference Engineering* Ch 2.2.1–2.2.3 (pp. 49–53) — Philip Kiely, Baseten Books 2026
- Vaswani et al., "Attention Is All You Need" (2017)
- Dao et al., "FlashAttention" (2022)